# Dataset 2 - EMD-Based IMF Time-Series Representation with LSTM for Schizophrenia Detection

This notebook investigates schizophrenia detection on Dataset 2 using a sequential representation of EEG signals derived from Empirical Mode Decomposition (EMD).

For each EEG segment, IMFs extracted from all channels are reorganized into a multivariate time-series representation and used as input to an LSTM network.

In [2]:
import os
import zipfile
import numpy as np
import mne
from glob import glob
import matplotlib.pyplot as plt
import seaborn as sns
from PyEMD import EMD
from scipy.signal import resample as sciresample
from scipy.stats import entropy
from tensorflow.keras.models import Sequential
import tensorflow as tf
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Dropout, BatchNormalization, Flatten, Dense,LSTM, GRU
from tensorflow.keras.optimizers import Adam
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import KFold, cross_val_score

In [3]:
subsets = ["subset_1.zip", "subset_2.zip", "subset_3.zip"]
base_dir = "/content"

for subset_zip in subsets:
    with zipfile.ZipFile(os.path.join(base_dir, subset_zip), 'r') as zip_ref:
        zip_ref.extractall(base_dir)
        print(f"Contenu extrait pour {subset_zip}")

Contenu extrait pour subset_1.zip
Contenu extrait pour subset_2.zip
Contenu extrait pour subset_3.zip


In [4]:
def extract_category_from_gnr(gnr_path):
    try:
        with open(gnr_path, 'r') as f:
            for line in f:
                if line.startswith("category="):
                    cat_value = line.split("=")[1].strip()
                    if cat_value in ["Patient", "Control"]:
                        return cat_value
        return None
    except Exception as e:
        print(f"Erreur lecture {gnr_path}: {e}")
        return None

In [5]:
def unify_channels(signal, desired_channels=24):

    n_channels, n_times = signal.shape
    if n_channels < desired_channels:
        new_signal = np.zeros((desired_channels, n_times), dtype=signal.dtype)
        new_signal[:n_channels, :] = signal
        return new_signal
    elif n_channels > desired_channels:
        print(f"Tronquage de {n_channels} canaux à {desired_channels} canaux.")
        return signal[:desired_channels, :]
    else:
        return signal

In [6]:
def my_resample(signal, old_fs, target_fs=250):
    if abs(old_fs - target_fs) < 1e-9:
        return signal, old_fs

    n_channels, n_times_old = signal.shape
    n_times_new = int(n_times_old * (target_fs / old_fs))

    resampled = np.zeros((n_channels, n_times_new), dtype=signal.dtype)
    for ch in range(n_channels):
        resampled[ch, :] = sciresample(signal[ch, :], n_times_new)

    return resampled, target_fs

In [7]:
def segment_signal(signal, fs, window_size=10, overlap=0.5):

    window_samples = int(window_size * fs)
    step = int(window_samples * (1 - overlap))
    segments = []
    for start in range(0, signal.shape[1] - window_samples + 1, step):
        seg = signal[:, start:start+window_samples]
        segments.append(seg)
    return segments

In [8]:
def emd_channel(channel_data, max_imf=5):

    emd = EMD()
    imfs = emd(channel_data)
    if imfs.shape[0] > max_imf:
        imfs = imfs[:max_imf, :]
    return imfs

In [ ]:
def differential_entropy_gaussian(signal_1d, eps=1e-12):

    var = np.var(signal_1d) + eps 
    de = 0.5 * np.log(2 * np.pi * np.e * var)
    return de

In [ ]:
def emd_extract_features_segment(segment, max_imf=5):
    n_channels, n_times = segment.shape
    feats = []

    for ch in range(n_channels):
        imfs = emd_channel(segment[ch, :], max_imf=max_imf)
        n_imf_ch, _ = imfs.shape

        # On limite ou on pad pour avoir "max_imf" IMF
        padded_imfs = np.zeros((max_imf, n_times))
        padded_imfs[:n_imf_ch, :] = imfs

        for i_imf in range(max_imf):
            imf_1d = padded_imfs[i_imf, :]
            de = differential_entropy_gaussian(imf_1d)
            feats.append(de)

    return np.array(feats)

In [ ]:
def build_dataset(base_dir,
                  subsets=["subset_1","subset_2","subset_3"],
                  desired_channels=24,
                  target_fs=256,
                  window_size=10,
                  overlap=0.5,
                  max_imf=5,
                 ):

    X = []
    y = []

    subfolders = ["1","2","3"]

    for subset_name in subsets:
        subset_folder = os.path.join(base_dir, subset_name)
        if not os.path.isdir(subset_folder):
            print(f"{subset_folder} n'existe pas. Ignoré.")
            continue


        for subject in os.listdir(subset_folder):
            subject_dir = os.path.join(subset_folder, subject)
            if not os.path.isdir(subject_dir):
                continue

            category = None
            for sf in subfolders:
                sf_path = os.path.join(subject_dir, sf)
                if not os.path.isdir(sf_path):
                    continue

                gnr_file = None
                for f in os.listdir(sf_path):
                    if f.lower().endswith(".gnr"):
                        gnr_file = os.path.join(sf_path, f)
                        break
                if gnr_file:
                    cat = extract_category_from_gnr(gnr_file)
                    if cat in ["Patient","Control"]:
                        category = cat
                        break

            if category is None:
                continue

            label = 1 if category == "Patient" else 0

            for sf in subfolders:
                sf_path = os.path.join(subject_dir, sf)
                if not os.path.isdir(sf_path):
                    continue

                for f in os.listdir(sf_path):
                    if f.lower().endswith(".edf"):
                        edf_path = os.path.join(sf_path, f)
                        try:

                            raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)
                            old_fs = raw.info['sfreq']
                            signal = raw.get_data()

                            # Unify :24 channels
                            signal_24 = unify_channels(signal, desired_channels)

                            # Resample :256 Hz
                            signal_rs, fs_rs = my_resample(signal_24, old_fs, target_fs)

                            segments = segment_signal(signal_rs, fs_rs,
                                                      window_size=window_size,
                                                      overlap=overlap)

                            # EMD + Entropie Diff
                            for seg in segments:
                                feats = emd_extract_features_segment(seg,
                                                                      max_imf=max_imf,
                                                                      )
                                X.append(feats)
                                y.append(label)

                        except Exception as e:
                            print(f"Erreur lecture {edf_path}: {e}")

    X = np.array(X)
    y = np.array(y)
    return X, y

In [ ]:
X, y = build_dataset(
    base_dir=base_dir,
    subsets=["subset_1","subset_2","subset_3"],
    desired_channels=24,
    target_fs=256,
    window_size=10,
    overlap=0.5,
    max_imf=5,

)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (3805, 120)
y shape: (3805,)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train set :", X_train.shape, y_train.shape)
print("Test set  :", X_test.shape, y_test.shape)

Train set : (3044, 120) (3044,)
Test set  : (761, 120) (761,)


**LSTM**

In [ ]:
def emd_imfs_to_timeseries(segment, max_imf=7):

    n_channels, n_times = segment.shape

    all_imfs = []

    for ch in range(n_channels):
        imfs = emd_channel(segment[ch, :], max_imf=max_imf)  # (n_imf_ch, n_times)
        n_imf_ch, _ = imfs.shape

        # Padding si < max_imf
        if n_imf_ch < max_imf:
            padded_imfs = np.zeros((max_imf, n_times))
            padded_imfs[:n_imf_ch, :] = imfs
        else:
            padded_imfs = imfs

        # On empile ces IMF dans la dimension "canaux"
        all_imfs.append(padded_imfs)

    # all_imfs = [ (max_imf, n_times) pour chaque canal ]
    # On concatène sur l'axe 0 => (n_channels*max_imf, n_times)
    all_imfs = np.concatenate(all_imfs, axis=0)

    # On veut (n_times, n_channels*max_imf) => on transpose
    all_imfs = all_imfs.T  

    return all_imfs

In [ ]:
def build_dataset_for_lstm(base_dir,
                           subsets=["subset_1","subset_2","subset_3"],
                           desired_channels=24,
                           target_fs=256,
                           window_size=10,
                           overlap=0.5,
                           max_imf=7):

    X_list = []
    y_list = []

    subfolders = ["1","2","3"]

    for subset_name in subsets:
        subset_folder = os.path.join(base_dir, subset_name)
        if not os.path.isdir(subset_folder):
            print(f"{subset_folder} n'existe pas. Ignoré.")
            continue

        for subject in os.listdir(subset_folder):
            subject_dir = os.path.join(subset_folder, subject)
            if not os.path.isdir(subject_dir):
                continue

            category = None
            for sf in subfolders:
                sf_path = os.path.join(subject_dir, sf)
                if not os.path.isdir(sf_path):
                    continue
                gnr_file = None
                for f in os.listdir(sf_path):
                    if f.lower().endswith(".gnr"):
                        gnr_file = os.path.join(sf_path, f)
                        break
                if gnr_file:
                    cat = extract_category_from_gnr(gnr_file)
                    if cat in ["Patient","Control"]:
                        category = cat
                        break

            if category is None:
                continue
            label = 1 if category == "Patient" else 0

            for sf in subfolders:
                sf_path = os.path.join(subject_dir, sf)
                if not os.path.isdir(sf_path):
                    continue

                for f in os.listdir(sf_path):
                    if f.lower().endswith(".edf"):
                        edf_path = os.path.join(sf_path, f)
                        try:
                            raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)
                            old_fs = raw.info['sfreq']
                            signal = raw.get_data() 

                            signal_24 = unify_channels(signal, desired_channels)
                            signal_rs, fs_rs = my_resample(signal_24, old_fs, target_fs)

                            segments = segment_signal(signal_rs, fs_rs,
                                                      window_size=window_size,
                                                      overlap=overlap)

                            for seg in segments:
                                timeseries_imfs = emd_imfs_to_timeseries(seg, max_imf=max_imf)
                                X_list.append(timeseries_imfs)
                                y_list.append(label)

                        except Exception as e:
                            print(f"Erreur lecture {edf_path}: {e}")


    X = np.array(X_list, dtype=object)
    y = np.array(y_list)

    return X, y

In [11]:
def build_lstm_model(input_shape):

    model = Sequential()
    model.add(LSTM(64, return_sequences=False, input_shape=input_shape))
    model.add(Dropout(0.3))
    model.add(Dense(32, activation='relu'))
    model.add(Dropout(0.3))
    model.add(Dense(1, activation='sigmoid'))

    model.compile(
        loss='binary_crossentropy',
        optimizer=Adam(learning_rate=1e-3),
        metrics=['accuracy']
    )
    return model

In [ ]:
base_dir = "/content"
X_list, y = build_dataset_for_lstm(
    base_dir=base_dir,
    subsets=["subset_1","subset_2","subset_3"],
    desired_channels=24,
    target_fs=256,
    window_size=10,
    overlap=0.5,
    max_imf=7
)


X = np.stack(X_list, axis=0)
print("X.shape =", X.shape)
print("y.shape =", y.shape)

In [ ]:
model = build_lstm_model(input_shape=(X.shape[1], X.shape[2]))
model.summary()

history = model.fit(
    X, y,
    epochs=100,
    batch_size=64,
    validation_split=0.2
)


loss, acc = model.evaluate(X, y, verbose=0)
print(f"Accuracy finale sur l'ensemble (train) : {acc*100:.2f}%")
plt.figure(figsize=(8, 4))
plt.plot(history.history['loss'], label='Training loss')
plt.plot(history.history['val_loss'], label='Validation loss')
plt.title('Courbe de la fonction de perte')
plt.xlabel('Épochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(history.history['accuracy'], label='Training accuracy')
plt.plot(history.history['val_accuracy'], label='Validation accuracy')
plt.title('Courbe de l’accuracy')
plt.xlabel('Épochs')
plt.ylabel('Accuracy')
plt.legend()
plt.show()